#### Task 1: File ingestion and regular expressions


In [12]:
from pathlib import Path

def robust_reader_function(fil_vei: str | Path):
    sti = Path(fil_vei)
    try:
        #Tries to read the info with utf-8
        file_text_content = sti.read_text(encoding="utf-8")
        encoding_used = "Utf-8"
        return (file_text_content, encoding_used)
    except UnicodeDecodeError: #If the utf-8 dont work it tries with iso
        file_text_content = sti.read_text(encoding="iso-8859-1")
        encoding_used = "iso-8859-1"
        return (file_text_content, encoding_used)


In [13]:
import re

def log_parser_function(data):
    valid_transactions = [] # Created an empty list you can put the valid transactions in
    for line in data:
        mønster = r"\[(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2})\] \[(?P<pos>\w+)\] item=(?P<vare>[a-zA-Z]+) price=(?P<pris>[\d\.]+) (?P<valuta>[A-Z]{3})"  
        resultat = re.search(mønster, line)
        # Adds the transactions to the empty list after checking that we get a result when running them
        if resultat:
            transaksjon = resultat.grupdict() #Turns them into a dictionary
            valid_transactions.append(transaksjon)
            return valid_transactions
        if not resultat: #Skips lines that doesent match the pattern I defined
            continue
    

##### Task 1 - Dissussion
In this task I created 2 functions. One for checking what kid ig encoding the data uses, and one to find the valid sales among mixed in with the system noise. 
The first function uses "try" to check the encoding. If the result after the first attempt is the spesific errorcode it goes on to try the next encoding. 
The second function takes each line in the data, checks it for the correct pattern and then adds it to the list of valid transactions. If the line doesnt match the pattern (system noise) the if-loops simply skips it.


#### Task 2 
1. Write a recursive directory search function that finds all log files in a nested directory
and returns a list of pathlib.Path objects

In [14]:
from pathlib import Path

def find_log_files(directory: str) -> list[Path]:
    """Find all log files in the given directory."""
    directory = Path(directory)
    log_files = []
    for item in directory.iterdir():
        if item.is_file() and item.suffix == ".log":
            log_files.append(item)
        elif item.is_dir():
            log_files.extend(find_log_files(item))
    return log_files

2. Write a recursive function that:

Traverses any combination of nested dictionaries and lists found in this file.

Returns a set (type hint: set[str] ) of all register id strings where "active":
true .

In [15]:
def find_active_registers(data) -> set[str]:
    """Finds all active registers in the given data structure."""
    active_registers = set()

    if isinstance(data, dict):
        if data.get("active") is True:
            register_id = data.get("id")
            if register_id:
                active_registers.add(register_id)

        for key, value in data.items():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for entry in data:
            active_registers.update(find_active_registers(entry))

    return active_registers
    
       

##### Task 2 - Discussion
In this task I created two functions. The first function finds all log files and returns a list of paths. It iterates through the folders and files, using recursion to search through subdirectories, and adds all files with the `.log` extension to the list.

The second function I created is a function that finds all active register IDs in the nested JSON data. It recursively traverses dictionaries and lists, checks which registers have "active" set to True, and adds their IDs to a set. Finally, it returns a set containing all unique active register IDs.

#### Task 3: Functions for cleaning and analysing data using NumPy

In [16]:
import numpy as np

def remowing_price_outliers(pris_data):
    #Konverter data til et 1d numpy array
    prisene = np.array(pris_data)
    # Trenger gjennomsnitt og standardavvik/sigma?
    my = np.mean(prisene)
    sigma = np.std(prisene)
    # Fjerner veldig høye verdier
    gyldige_priser = []
    for pris in prisene: 
        avstand = abs(pris - my)
        if avstand <= (3*sigma):
            gyldige_priser.append(pris)

    #må finne medianen til de gyldige prisene
    median_verdi = np.median(gyldige_priser)
    # de høyeste veriene som skal erstattes skal byttes ut med den medianen
    for i in range(len(prisene)):
        avstand = abs(prisene[i] - my)
        if avstand <= (3*sigma) or prisene[i] != 999.00:
            prisene[i] = median_verdi
    return prisene

In [17]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    # Regner ut antall refusjoner? 
    
    return refund_count / (sales_count + 10**(-6))
 

##### Task 3 - Discussion

#### Task 4

1. Write a function that:

Converts the list of transaction dictionaries read from the log files into a
pandas.DataFrame.

Converts the timestamp column to datetime and sets it as the DataFrame
index.

Sorts the DataFrame chronologically by timestamp.

In [18]:
import pandas as pd

def analyze_register_data(data: list[dict]) -> pd.DataFrame:
    """Convert transaction data into a DataFrame sorted by timestamp."""
    df = pd.DataFrame(data)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()
    return df

2. Write a function that has the DataFrame you just created as input, groups data by
item and resamples transactions in 1-hour windows (1h), calculating total price
(sum).

In [19]:
def calculate_hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Calculate hourly revenue for each item."""
    hourly_revenue = df.groupby("item").resample("1h")["price"].sum()
    return hourly_revenue

3. Write a function that has the DataFrame as input and creates a pivot table displaying
total transaction revenue grouped by item across status categories ( SUCCESS ,
REFUND , CANCELLED )

In [20]:
def create_status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Create a pivot table showing total revenue for each item by status."""
    pivot_table = df.pivot_table(index="item", columns="status", values="price", aggfunc="sum", fill_value=0)
    return pivot_table

4. Write a function that has the DataFrame as input and calculates the refund ratio per
item by calling the function you made in task 3.

In [21]:
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item."""
    item_counts = df.groupby("item")["status"].value_counts().unstack(fill_value=0)

    sales_counts = item_counts.get("SUCCESS", pd.Series(0, index=item_counts.index)).to_numpy() #hvis numpy i task 3
    refund_counts = item_counts.get("REFUND", pd.Series(0, index=item_counts.index)).to_numpy()

    item_counts["refund_ratio"] = remowing_price_outliers( 
        sales_counts,
        refund_counts
    )

    return item_counts[["refund_ratio"]]

#### Task 4 - Discussion

#### Task 5:  Create full workflow


In [22]:
def full_workflow(data):
    #1. From task 2- finds logfiles and active registers (is that supposed to refrence the json-file?)
    log_files = find_log_files(data)
    active_registers = find_active_registers(data) #json-file?

    #2. Takes text logs and parse records and turns them into a list of dictionaries
    log_dictionary = log_parser_function(log_files)

    #3.From task 4. Takes the finhised log-files from above and converts them to PandasDataframes
    PaDa = analyze_register_data(log_dictionary)

    #4. From task 3. Removes outliers
    cleaned_prices = remowing_price_outliers(PaDa)

    #5. Currency correction
    





1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove price glitches.
5. Apply currency correction: multiply price by for all transactions where
currency == "NOK" .
6. Generate resampled reports, pivot tables and refund ratios

Outputs (files your code should generate)
cleaned_transactions.csv : A CSV table containing all parsed, outlier-cleaned,
and currency-corrected transactions.
hourly_revenue.json : A JSON report containing total revenue per hour per item.
status_pivot_table.csv : A CSV file containing a pivot table of total transaction
revenue grouped by item across status categories.
item_refund_ratios.csv : A CSV table containing refund ratios for each item
type.

 Oversette task 3
 Markdown på task 3